# ST-CDGM — Test d'intervention causale sur `A_dag`

**Objectif** : vérifier empiriquement, sur le checkpoint courant, si la matrice DAG
apprise (`rcn_cell.A_dag`) conditionne réellement les prédictions du décodeur de
diffusion. C'est un test de type **do-calculus** : on fige tout (données, bruit de
sampling, poids), on ne fait varier **que** `A_dag`, et on mesure la différence
pixel-par-pixel entre les sorties.

## Protocole

Pour un même échantillon test et un même `torch.Generator(seed=...)`, on génère
trois prédictions sous trois régimes de `A_dag` :

| Régime | `A_dag` effectif | Interprétation |
|---|---|---|
| `orig` | matrice apprise (baseline) | prédiction normale |
| `zero` | tous les poids à 0 | DAG "éteint", aucune relation causale |
| `perm` | lignes permutées (seed fixe) | DAG existant mais incohérent |

## Verdict

- Si `‖pred_orig − pred_zero‖ ≈ 0` **et** `‖pred_orig − pred_perm‖ ≈ 0`
  → la DAG n'affecte pas la prédiction. Le conditionnement passe uniquement par
  l'état `H_T` (et le projecteur spatial, qui lui ne voit pas `A_dag`).
  **Conclusion : le modèle n'est pas causalement conditionné par la DAG.**
- Si les deltas sont structurés (non-nuls, MSE mesurable) et que `zero` et `perm`
  dégradent davantage que `orig`
  → la DAG a un effet causal réel.

## Pré-requis

- `models/st_cdgm_checkpoint.pth` (checkpoint 10-epoch Sprint 1).
- `data/raw/test/EC-Earth3_*_compressed.nc` (jeu de test).
- `config/training_config.yaml`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from omegaconf import OmegaConf

project_root = Path.cwd()
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

CONFIG = OmegaConf.load("config/training_config.yaml")
DEVICE = torch.device("cpu")

CHECKPOINT_PATH = Path("models/st_cdgm_checkpoint.pth")
assert CHECKPOINT_PATH.is_file(), f"Checkpoint introuvable: {CHECKPOINT_PATH}"

print(f"Device: {DEVICE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

In [ ]:
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from st_cdgm.models.intelligible_encoder import (
    IntelligibleVariableConfig,
    IntelligibleVariableEncoder,
    SpatialConditioningProjector,
    CausalConditioningProjector,
)
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.diffusion_decoder import CausalDiffusionDecoder

try:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
except TypeError:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

required = {"encoder_state_dict", "rcn_cell_state_dict", "diffusion_state_dict"}
missing = required - set(checkpoint.keys())
if missing:
    raise KeyError(f"Checkpoint invalide, cles manquantes: {missing}")

print(f"Epoch checkpoint: {checkpoint.get('epoch', 'n/a')}")
print(f"Cles presentes: {sorted(checkpoint.keys())}")

In [ ]:
ckpt_cfg = checkpoint.get("config", {})
ckpt_cfg_full = checkpoint.get("config_full", {})

SEQ_LEN = int(ckpt_cfg.get("seq_len", CONFIG.data.seq_len))
STATIC_PATH = str(CONFIG.data.static_path) if CONFIG.data.get("static_path") else None

norm_cfg = ckpt_cfg.get("normalization", {})
MEAN_PATH = str(norm_cfg.get("mean_path", "data/raw/normalization_coefs/mean_1974_2011.nc"))
STD_PATH = str(norm_cfg.get("std_path", "data/raw/normalization_coefs/std_1974_2011.nc"))

LR_VARIABLES = list(ckpt_cfg.get("lr_variables", CONFIG.data.lr_variables))
STATIC_VARIABLES = list(ckpt_cfg.get("static_variables", CONFIG.data.static_variables))
TEST_HR_VARIABLES = ["pr"]

TEST_GCM = "EC-Earth3"
TEST_ROOT = Path("data/raw/test")
lr_path = TEST_ROOT / f"{TEST_GCM}_histupdated_compressed.nc"
hr_path = TEST_ROOT / f"{TEST_GCM}_historical_precip_compressed.nc"

pipeline = NetCDFDataPipeline(
    lr_path=str(lr_path),
    hr_path=str(hr_path),
    static_path=STATIC_PATH,
    seq_len=SEQ_LEN,
    baseline_strategy=CONFIG.data.baseline_strategy,
    baseline_factor=CONFIG.data.get("baseline_factor", 4),
    normalize=bool(CONFIG.data.normalize),
    nan_fill_strategy=CONFIG.data.nan_fill_strategy,
    precipitation_delta=CONFIG.data.get("precipitation_delta", 0.01),
    lr_variables=LR_VARIABLES,
    hr_variables=TEST_HR_VARIABLES,
    static_variables=STATIC_VARIABLES,
    means_path=MEAN_PATH if Path(MEAN_PATH).exists() else None,
    stds_path=STD_PATH if Path(STD_PATH).exists() else None,
)

dataset = pipeline.build_sequence_dataset(seq_len=SEQ_LEN, as_torch=True)
print(f"Dataset pret pour GCM={TEST_GCM}.")

In [ ]:
lr_shape = tuple(ckpt_cfg.get("lr_shape", CONFIG.graph.lr_shape))
hr_shape = tuple(ckpt_cfg.get("hr_shape", CONFIG.graph.hr_shape))
include_mid_layer = bool(ckpt_cfg_full.get("graph", {}).get("include_mid_layer", CONFIG.graph.include_mid_layer))

builder = HeteroGraphBuilder(
    lr_shape=lr_shape,
    hr_shape=hr_shape,
    static_dataset=pipeline.get_static_dataset(),
    include_mid_layer=include_mid_layer,
)

allowed_nodes = set(builder.dynamic_node_types) | set(builder.static_node_types)
encoder_configs = [
    IntelligibleVariableConfig(
        name=mp.name,
        meta_path=(mp.src, mp.relation, mp.target),
        pool=mp.get("pool", "mean"),
    )
    for mp in CONFIG.encoder.metapaths
    if mp.src in allowed_nodes and mp.target in allowed_nodes
]
if pipeline.get_static_dataset() is not None:
    encoder_configs.append(
        IntelligibleVariableConfig(
            name="static",
            meta_path=("SP_HR", "causes", "GP850"),
            pool="mean",
        )
    )

probe_sample = next(iter(dataset))
hr_channels = probe_sample["residual"].shape[1]
rcn_driver_dim = probe_sample["lr"].shape[1]

encoder = IntelligibleVariableEncoder(
    configs=encoder_configs,
    hidden_dim=CONFIG.encoder.hidden_dim,
    conditioning_dim=CONFIG.encoder.conditioning_dim,
).to(DEVICE)

rcn_cell = RCNCell(
    num_vars=len(encoder_configs),
    hidden_dim=CONFIG.rcn.hidden_dim,
    driver_dim=rcn_driver_dim,
    reconstruction_dim=rcn_driver_dim,
    dropout=CONFIG.rcn.dropout,
).to(DEVICE)
rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get("detach_interval"))

diffusion = CausalDiffusionDecoder(
    in_channels=hr_channels,
    conditioning_dim=CONFIG.diffusion.conditioning_dim,
    height=CONFIG.diffusion.height,
    width=CONFIG.diffusion.width,
    num_diffusion_steps=CONFIG.diffusion.steps,
    unet_kwargs=dict(
        layers_per_block=1,
        block_out_channels=(32, 64),
        down_block_types=("DownBlock2D", "CrossAttnDownBlock2D"),
        up_block_types=("CrossAttnUpBlock2D", "UpBlock2D"),
        mid_block_type="UNetMidBlock2D",
        norm_num_groups=8,
        class_embed_type="projection",
        projection_class_embeddings_input_dim=len(encoder_configs) * CONFIG.diffusion.conditioning_dim,
        resnet_time_scale_shift="scale_shift",
        attention_head_dim=32,
        only_cross_attention=[False, True],
    ),
).to(DEVICE)

try:
    from st_cdgm.utils.checkpoint import strip_torch_compile_prefix
except ModuleNotFoundError:
    def strip_torch_compile_prefix(sd):
        if not sd or not any(str(k).startswith("_orig_mod.") for k in sd.keys()):
            return sd
        return {
            (k[len("_orig_mod."):] if str(k).startswith("_orig_mod.") else k): v
            for k, v in sd.items()
        }

encoder.load_state_dict(strip_torch_compile_prefix(checkpoint["encoder_state_dict"]))
rcn_cell.load_state_dict(strip_torch_compile_prefix(checkpoint["rcn_cell_state_dict"]))
diffusion.load_state_dict(strip_torch_compile_prefix(checkpoint["diffusion_state_dict"]))

spatial_projector = None
_sp_state = checkpoint.get("spatial_projector_state_dict") or checkpoint.get("spatial_projector_state")
if _sp_state is not None:
    _sp_clean = strip_torch_compile_prefix(_sp_state)
    _target_shape = tuple(CONFIG.diffusion.get("spatial_target_shape", [6, 7]))
    _cond_dim = int(CONFIG.diffusion.conditioning_dim)
    # Sprint 2 checkpoints wrap the spatial path as ``spatial.*`` and add ``dag_mlp`` / ``dag_norm``.
    if "dag_mlp.0.weight" in _sp_clean:
        w_out = _sp_clean["dag_mlp.2.weight"]
        if int(w_out.shape[0]) % _cond_dim != 0:
            raise RuntimeError(
                f"Incompatible causal projector head: dag_mlp.2 out_dim={w_out.shape[0]} "
                f"not divisible par conditioning_dim={_cond_dim}"
            )
        _ndt = int(w_out.shape[0]) // _cond_dim
        spatial_projector = CausalConditioningProjector(
            num_vars=len(encoder_configs),
            hidden_dim=CONFIG.rcn.hidden_dim,
            conditioning_dim=_cond_dim,
            lr_shape=lr_shape,
            target_shape=_target_shape,
            num_dag_tokens=_ndt,
        ).to(DEVICE)
        spatial_projector.load_state_dict(_sp_clean, strict=True)
        spatial_projector.eval()
        print(
            f"CausalConditioningProjector charge (target_shape={_target_shape}, num_dag_tokens={_ndt})."
        )
    else:
        spatial_projector = SpatialConditioningProjector(
            num_vars=len(encoder_configs),
            hidden_dim=CONFIG.rcn.hidden_dim,
            conditioning_dim=_cond_dim,
            lr_shape=lr_shape,
            target_shape=_target_shape,
        ).to(DEVICE)
        spatial_projector.load_state_dict(_sp_clean, strict=True)
        spatial_projector.eval()
        print(f"SpatialConditioningProjector charge (target_shape={_target_shape}).")
else:
    print("[INFO] Pas de spatial_projector dans le checkpoint — fallback global.")

encoder.eval(); rcn_cell.eval(); diffusion.eval()
print(f"Modeles charges | q={len(encoder_configs)} | A_dag shape={tuple(rcn_cell.A_dag.shape)}")

## Diagnostic préalable de `A_dag`

Avant d'intervenir, on inspecte la matrice apprise : norme, sparsité, top-k
relations. Une DAG quasi-nulle ou quasi-uniforme rendrait le test d'intervention
moins informatif (l'édition produit un delta par rapport à une référence déjà
dégénérée).

In [ ]:
with torch.no_grad():
    A = rcn_cell.A_dag.detach().cpu().clone()
    A_masked = A - torch.diag(torch.diagonal(A))

var_names = [c.name for c in encoder_configs]
print(f"A_dag.shape = {tuple(A.shape)}")
print(f"  |A|_F      = {A.norm().item():.4f}")
print(f"  max|A|     = {A.abs().max().item():.4f}")
print(f"  mean|A|    = {A.abs().mean().item():.4f}")
print(f"  sparsite   = {(A.abs() < 1e-3).float().mean().item() * 100:.1f}% entrees |.| < 1e-3")

# Top-5 aretes par magnitude
flat = A_masked.abs().flatten()
topk = torch.topk(flat, k=min(5, flat.numel()))
print("\nTop-5 aretes (i -> j, |A_ij|):")
for rank, (val, idx) in enumerate(zip(topk.values.tolist(), topk.indices.tolist())):
    i, j = divmod(idx, A.shape[1])
    src = var_names[i] if i < len(var_names) else f"v{i}"
    tgt = var_names[j] if j < len(var_names) else f"v{j}"
    print(f"  {rank + 1}. {src:20s} -> {tgt:20s}  |A|={val:.4f}")

A_df = pd.DataFrame(A_masked.numpy(), index=var_names, columns=var_names)
A_df

## Helpers d'inférence sous `A_dag` édité

On remplace temporairement `rcn_cell.A_dag` par une version éditée (contexte
manager). Le projecteur spatial et le conditionnement `H_T` sont recalculés à
chaque régime parce que `H_T` dépend de `A_dag` via la boucle RCN. C'est
précisément cette dépendance qu'on teste.

In [ ]:
import contextlib

@contextlib.contextmanager
def override_A_dag(cell, new_A):
    """Temporarily replace ``cell.A_dag`` parameter data with ``new_A``.

    Restores the original value on exit. No gradient is taken during the
    intervention; we only need forward-pass numerical effects.
    """
    original = cell.A_dag.data.clone()
    try:
        cell.A_dag.data.copy_(new_A.to(cell.A_dag.device, cell.A_dag.dtype))
        yield
    finally:
        cell.A_dag.data.copy_(original)


def make_edited_A_dag(original: torch.Tensor, regime: str, perm_seed: int = 0) -> torch.Tensor:
    """Build the edited A_dag matrix for a given intervention regime."""
    if regime == "orig":
        return original.clone()
    if regime == "zero":
        return torch.zeros_like(original)
    if regime == "perm":
        g = torch.Generator().manual_seed(perm_seed)
        perm = torch.randperm(original.shape[0], generator=g)
        return original[perm, :].clone()
    if regime == "negate":
        return -original.clone()
    raise ValueError(f"regime inconnu: {regime}")


def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample["lr"]
    seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    dynamic_features = {nt: lr_nodes_steps[0] for nt in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)
    out = {
        "lr": lr_tensor,
        "residual": sample["residual"],
        "baseline": sample.get("baseline"),
        "hetero": hetero,
    }
    return out


@torch.no_grad()
def run_inference_under_regime(sample, regime: str, sample_seed: int, num_steps: int = 15):
    """Return the predicted t_mean (HR field) under the given A_dag regime.

    `sample_seed` fixes the diffusion noise, so the *only* source of
    between-regime variability is A_dag itself.
    """
    batch = convert_sample_to_batch(sample, builder, DEVICE)
    lr_data = batch["lr"].to(DEVICE)
    baseline = batch["baseline"]
    if baseline is None:
        baseline = torch.zeros_like(batch["residual"])
    baseline_t = baseline[-1].to(DEVICE).unsqueeze(0)

    edited = make_edited_A_dag(rcn_cell.A_dag.data, regime)
    with override_A_dag(rcn_cell, edited):
        H_init = encoder.init_state(batch["hetero"]).to(DEVICE)
        drivers = [lr_data[t] for t in range(lr_data.shape[0])]
        seq_out = rcn_runner.run(H_init, drivers, reconstruction_sources=None)

        H_last = seq_out.states[-1]
        A_last = seq_out.dag_matrices[-1]
        conditioning = encoder.project_state_tensor(H_last).to(DEVICE)
        if spatial_projector is None:
            conditioning_spatial = None
        elif hasattr(spatial_projector, "num_dag_tokens"):
            conditioning_spatial = spatial_projector(H_last, A_last).to(DEVICE)
        else:
            conditioning_spatial = spatial_projector(H_last).to(DEVICE)

        gen = torch.Generator(device="cpu").manual_seed(sample_seed)
        generated = diffusion.sample(
            conditioning,
            num_steps=num_steps,
            scheduler_type=CONFIG.diffusion.get("scheduler_type", "ddpm"),
            apply_constraints=False,
            baseline=baseline_t,
            conditioning_spatial=conditioning_spatial,
            generator=gen,
        )

    target_batch = baseline_t + batch["residual"][-1].to(DEVICE).unsqueeze(0)

    pred = generated.t_mean
    if pred.shape != target_batch.shape:
        pred = F.interpolate(pred, size=target_batch.shape[-2:], mode="bicubic", align_corners=False).clamp(min=0.0)
    pred = torch.nan_to_num(pred, nan=0.0, posinf=0.0, neginf=0.0)
    target_batch = torch.nan_to_num(target_batch, nan=0.0, posinf=0.0, neginf=0.0)

    return {
        "pred": pred.cpu(),
        "target": target_batch.cpu(),
        "H_last": H_last.cpu(),
        "conditioning": conditioning.cpu(),
        "conditioning_spatial": conditioning_spatial.cpu() if conditioning_spatial is not None else None,
    }

print("Helpers prets.")

## Exécution du test d'intervention

On itère sur `N_SAMPLES` échantillons test. Pour chacun, on lance 3 inférences
(un par régime) avec **le même seed de bruit** pour que seul `A_dag` varie.

In [ ]:
N_SAMPLES = 5
NUM_STEPS = int(CONFIG.diffusion.get("val_num_steps", 15))
REGIMES = ["orig", "zero", "perm"]

it = iter(dataset)
records = []
per_sample_outputs = []

for idx in range(N_SAMPLES):
    try:
        sample = next(it)
    except StopIteration:
        print(f"[WARN] Dataset epuise a {idx} echantillons.")
        break

    sample_seed = 1000 + idx  # different per sample, stable across regimes
    regime_outputs = {}
    for regime in REGIMES:
        out = run_inference_under_regime(sample, regime, sample_seed=sample_seed, num_steps=NUM_STEPS)
        regime_outputs[regime] = out

    per_sample_outputs.append(regime_outputs)

    target = regime_outputs["orig"]["target"]
    pred_orig = regime_outputs["orig"]["pred"]
    pred_zero = regime_outputs["zero"]["pred"]
    pred_perm = regime_outputs["perm"]["pred"]

    def _mse(a, b):
        return torch.mean((a - b) ** 2).item()

    def _mae(a, b):
        return torch.mean(torch.abs(a - b)).item()

    rec = {
        "sample_idx": idx,
        "sample_seed": sample_seed,
        # Prediction accuracy per regime (vs ground truth)
        "mse_orig_vs_target": _mse(pred_orig, target),
        "mse_zero_vs_target": _mse(pred_zero, target),
        "mse_perm_vs_target": _mse(pred_perm, target),
        # Interventional deltas (the key signal)
        "delta_orig_vs_zero_mse": _mse(pred_orig, pred_zero),
        "delta_orig_vs_perm_mse": _mse(pred_orig, pred_perm),
        "delta_orig_vs_zero_mae": _mae(pred_orig, pred_zero),
        "delta_orig_vs_perm_mae": _mae(pred_orig, pred_perm),
        # Relative magnitude of the delta vs the orig prediction itself
        "pred_orig_rms": float(pred_orig.pow(2).mean().sqrt()),
        "pred_orig_max": float(pred_orig.abs().max()),
    }
    records.append(rec)
    print(f"[{idx + 1}/{N_SAMPLES}] "
          f"d(orig,zero)={rec['delta_orig_vs_zero_mse']:.2e} | "
          f"d(orig,perm)={rec['delta_orig_vs_perm_mse']:.2e} | "
          f"MSE_orig={rec['mse_orig_vs_target']:.4f} "
          f"MSE_zero={rec['mse_zero_vs_target']:.4f}")

df = pd.DataFrame(records)
df

## Analyse et verdict

On compare la grandeur des deltas interventionnels à celle du signal `pred_orig`
lui-même (RMS). Un ratio `delta / rms(pred_orig) < 1%` signifie que
l'édition de `A_dag` ne change virtuellement rien : la DAG est décorative.

In [ ]:
summary = df.mean(numeric_only=True).to_dict()

# Relative delta vs signal magnitude (squared ratio → unitless)
rms_orig2 = float(df["pred_orig_rms"].mean() ** 2)
rel_zero = summary["delta_orig_vs_zero_mse"] / max(rms_orig2, 1e-12)
rel_perm = summary["delta_orig_vs_perm_mse"] / max(rms_orig2, 1e-12)

# Effet sur la qualité de prédiction (vs ground truth)
degrad_zero = summary["mse_zero_vs_target"] - summary["mse_orig_vs_target"]
degrad_perm = summary["mse_perm_vs_target"] - summary["mse_orig_vs_target"]

print("=" * 70)
print("TEST D'INTERVENTION CAUSALE SUR A_dag")
print("=" * 70)
print(f"\nEchantillons: {len(df)}")
print(f"Steps de sampling: {NUM_STEPS}")
print(f"Regimes testes: {REGIMES}")

print("\n--- Deltas interventionnels (moyennes) ---")
print(f"MSE(pred_orig, pred_zero) = {summary['delta_orig_vs_zero_mse']:.6e}")
print(f"MSE(pred_orig, pred_perm) = {summary['delta_orig_vs_perm_mse']:.6e}")
print(f"RMS(pred_orig)^2          = {rms_orig2:.6e}")
print(f"  => delta_zero / signal  = {rel_zero * 100:.3f} % (puissance relative)")
print(f"  => delta_perm / signal  = {rel_perm * 100:.3f} % (puissance relative)")

print("\n--- Degradation de la precision (MSE vs cible) ---")
print(f"MSE orig vs target  = {summary['mse_orig_vs_target']:.6f}")
print(f"MSE zero vs target  = {summary['mse_zero_vs_target']:.6f} (delta={degrad_zero:+.6f})")
print(f"MSE perm vs target  = {summary['mse_perm_vs_target']:.6f} (delta={degrad_perm:+.6f})")

print("\n" + "=" * 70)
print("VERDICT")
print("=" * 70)

THRESHOLD_PCT = 1.0  # 1% of signal power
if rel_zero * 100 < THRESHOLD_PCT and rel_perm * 100 < THRESHOLD_PCT:
    verdict = (
        "DAG NON-CONDITIONNANTE. Les interventions sur A_dag produisent un effet\n"
        f"inferieur a {THRESHOLD_PCT}% du signal. Le decodeur de diffusion ignore\n"
        "la structure causale: le conditionnement passe presque exclusivement par\n"
        "H_T (et le projecteur spatial), qui ne dependent que faiblement de A_dag\n"
        "parce que le grad gate est a 0 et detach_dag_in_state est True."
    )
    verdict_code = "non_conditioning"
elif rel_zero * 100 > THRESHOLD_PCT and degrad_zero > 0 and degrad_perm > 0:
    verdict = (
        "DAG CONDITIONNANTE. Les interventions produisent un effet mesurable et\n"
        "les editions degradent la precision. La structure causale affecte\n"
        "reellement les predictions — l'explicabilite causale (do-calculus) est\n"
        "operationnelle."
    )
    verdict_code = "conditioning"
else:
    verdict = (
        "RESULTAT AMBIGU. Les deltas sont non-negligeables mais la degradation de\n"
        "precision est inconsistante (zero ou perm n'empire pas systematiquement).\n"
        "Possible que la DAG affecte H_T mais sans direction causale utile —\n"
        "typique d'un regime ou gate=0 mais ou A_dag capture du bruit corrélé."
    )
    verdict_code = "ambiguous"

print(verdict)
print("\nCode: " + verdict_code)

## Visualisation : delta spatial d'une intervention

Carte de `|pred_orig − pred_zero|` sur le premier échantillon. Si la DAG est
conditionnante, on devrait voir une structure spatiale cohérente (patterns
régionaux). Si elle est décorative, la carte sera du bruit diffus.

In [ ]:
import matplotlib.pyplot as plt

if per_sample_outputs:
    outputs_0 = per_sample_outputs[0]
    pred_orig = outputs_0["orig"]["pred"][0, 0].numpy()
    pred_zero = outputs_0["zero"]["pred"][0, 0].numpy()
    pred_perm = outputs_0["perm"]["pred"][0, 0].numpy()
    target = outputs_0["orig"]["target"][0, 0].numpy()

    delta_zero = np.abs(pred_orig - pred_zero)
    delta_perm = np.abs(pred_orig - pred_perm)

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    vmax_pred = max(pred_orig.max(), target.max())
    vmax_delta = max(delta_zero.max(), delta_perm.max(), 1e-6)

    axes[0, 0].imshow(target, cmap="Blues", vmin=0, vmax=vmax_pred, aspect="auto")
    axes[0, 0].set_title("Target HR")
    axes[0, 1].imshow(pred_orig, cmap="Blues", vmin=0, vmax=vmax_pred, aspect="auto")
    axes[0, 1].set_title("Pred (A_dag original)")
    axes[0, 2].imshow(pred_zero, cmap="Blues", vmin=0, vmax=vmax_pred, aspect="auto")
    axes[0, 2].set_title("Pred (A_dag = 0)")

    im1 = axes[1, 0].imshow(delta_zero, cmap="Reds", vmin=0, vmax=vmax_delta, aspect="auto")
    axes[1, 0].set_title("|orig - zero|")
    plt.colorbar(im1, ax=axes[1, 0], fraction=0.046)

    im2 = axes[1, 1].imshow(delta_perm, cmap="Reds", vmin=0, vmax=vmax_delta, aspect="auto")
    axes[1, 1].set_title("|orig - perm|")
    plt.colorbar(im2, ax=axes[1, 1], fraction=0.046)

    axes[1, 2].axis("off")
    axes[1, 2].text(
        0.05,
        0.95,
        f"Signal RMS     = {np.sqrt((pred_orig**2).mean()):.4f}\n"
        f"max|orig-zero| = {delta_zero.max():.4f}\n"
        f"max|orig-perm| = {delta_perm.max():.4f}\n"
        f"mean|orig-zero|= {delta_zero.mean():.4f}\n"
        f"mean|orig-perm|= {delta_perm.mean():.4f}",
        fontfamily="monospace",
        fontsize=10,
        va="top",
        transform=axes[1, 2].transAxes,
    )

    plt.suptitle("Intervention causale sur A_dag — echantillon 0", fontsize=13)
    plt.tight_layout()
    plt.savefig("results/dag_intervention_sample0.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Figure sauvegardee: results/dag_intervention_sample0.png")

## Export des résultats

In [ ]:
import json

results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

csv_path = results_dir / "dag_intervention_metrics.csv"
df.to_csv(csv_path, index=False)

report = {
    "checkpoint": str(CHECKPOINT_PATH),
    "epoch": checkpoint.get("epoch", "n/a"),
    "n_samples": int(len(df)),
    "num_steps": int(NUM_STEPS),
    "regimes": REGIMES,
    "summary": {k: float(v) for k, v in summary.items()},
    "relative_delta_pct": {
        "orig_vs_zero": float(rel_zero * 100),
        "orig_vs_perm": float(rel_perm * 100),
    },
    "precision_degradation": {
        "zero_minus_orig": float(degrad_zero),
        "perm_minus_orig": float(degrad_perm),
    },
    "verdict_code": verdict_code,
    "verdict": verdict,
    "A_dag_stats": {
        "frobenius_norm": float(A.norm()),
        "max_abs": float(A.abs().max()),
        "mean_abs": float(A.abs().mean()),
    },
}

json_path = results_dir / "dag_intervention_report.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"Metriques CSV : {csv_path}")
print(f"Rapport JSON  : {json_path}")
print(f"\nVerdict final : {verdict_code.upper()}")